# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds
**Nguyễn Nhật Thắng · MSSV 2A202602727** · chạy trên **Google Colab (GPU T4)**

Quy tắc: chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4** (README mục 2.1–2.2).

**Cách chạy**
1. *Runtime → Change runtime type → T4 GPU*.
2. (Khuyên dùng) Upload `images.zip` (MD5 `b7b30f96d466fba86016aa5a26606e0f`) lên Google Drive vào `MyDrive/K4_Day2/images.zip`. Nếu không có, ô tải dữ liệu tự tải từ Zenodo rồi lưu một bản lên Drive.
3. *Runtime → Run all*. Mọi log, checkpoint, logit, đường cong, dự đoán đều ghi vào `MyDrive/K4_Day2/`.
   Nếu phiên bị ngắt: chạy lại *Run all*, các lần chạy đã xong được bỏ qua (`summary.json`), lần đang dở tiếp tục từ `last.pt`.
4. Cuối notebook: chép sản phẩm vào `submissions/2A202602727_NguyenNhatThang/` và tải về (hoặc push lên fork).

Bản đồ: Bước 0 (EDA, kiểm tra) → 1 (backbone) → 2 (công thức huấn luyện) → 3 (suy luận, độ trễ) → 4 (chung kết + `eval.py`) → 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab đã có torch, torchvision, sklearn, matplotlib)
!pip -q install timm openpyxl

import sys, os, platform, json, shutil, glob, time, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MSSV, HO_TEN = "2A202602727", "NguyenNhatThang"
GITHUB_USER = "nhat-thang"
# Tên fork trên GitHub. Nếu bạn đổi tên fork theo mẫu K4-DAY02-HoVaTen-MSSV thì sửa dòng này.
REPO_NAME = "K4-Track4-Day2-Deeplearning-Advance"
WORK = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = WORK / REPO_NAME
if not REPO_DIR.exists():
    !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q --ff-only

SUB_DIR = REPO_DIR / "submissions" / f"{MSSV}_{HO_TEN}"
CODE_DIR = SUB_DIR / "code"
sys.path.insert(0, str(REPO_DIR))   # eval.py
sys.path.insert(0, str(CODE_DIR))   # code/ của mình (starter đã hoàn thiện)

# Nơi lưu bền (Drive): checkpoint/log ở runs/, sản phẩm nộp ở submission/
DRIVE_ROOT = Path("/content/drive/MyDrive/K4_Day2") if IN_COLAB else WORK / "k4_day2_out"
RUNS = DRIVE_ROOT / "runs"
OUT = DRIVE_ROOT / "submission"
PRED, CURVES, FIGS, TABLES = OUT / "predictions", OUT / "curves", OUT / "figures", DRIVE_ROOT / "tables"
for d in (RUNS, PRED, CURVES, FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch, torchvision, timm, sklearn, matplotlib
import matplotlib.pyplot as plt
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
print("CPU cores:", os.cpu_count())

### Tải dữ liệu
Ưu tiên `MyDrive/K4_Day2/images.zip` (bản bạn đã tải); không có thì tải từ Zenodo. Kiểm tra MD5, giải nén vào đĩa cục bộ của Colab (nhanh hơn đọc từ Drive), tải 4 file CSV fold 0 từ GitHub của tác giả.

In [ ]:
DATA = Path("/content/data") if IN_COLAB else WORK / "data"
LABELS_DIR = DATA / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
ZIP = DATA / "images.zip"
DRIVE_ZIP = DRIVE_ROOT / "images.zip"

if not ZIP.exists():
    if DRIVE_ZIP.exists():
        print("Chép images.zip từ Drive ...")
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        print("Tải images.zip từ Zenodo ...")
        !wget -q -O {ZIP} "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open(ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
print("MD5 OK")
if not DRIVE_ZIP.exists():
    shutil.copy(ZIP, DRIVE_ZIP)   # lần sau khỏi tải lại

IMAGES_DIR = DATA / "images"
if not IMAGES_DIR.exists() or len(list(IMAGES_DIR.rglob("*.jpg"))) < 17509:
    !unzip -q -n {ZIP} -d {IMAGES_DIR}
jpgs = list(IMAGES_DIR.rglob("*.jpg"))
IMAGES_DIR = jpgs[0].parent          # thư mục thực sự chứa .jpg
print("IMAGES_DIR =", IMAGES_DIR, "|", len(jpgs), "ảnh")
!ls {IMAGES_DIR} | head -3

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not (LABELS_DIR / f"{name}.csv").exists():
        !wget -q -O {LABELS_DIR}/{name}.csv {BASE}/{name}.csv
!ls -la {LABELS_DIR}

### Kiểm tra code tự viết (focal γ=0 ≡ CE, CutMix, gộp BN, temperature, scheduler, EMA) và test của repo

In [ ]:
!cd {CODE_DIR} && python -m unittest test_code -v 2>&1 | tail -25
!cd {REPO_DIR} && python -m unittest discover -s tests 2>&1 | tail -5

In [ ]:
import importlib
import dataset as D, model as M, losses as L, train, inference as I, benchmark as BM, experiments as X
from train import Config, run, alias_run
from eval import compute_metrics, save_predictions, read_pred
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 12          # công thức nền: 12 epoch cho mọi thí nghiệm (GUIDE 1.4). Hết hạn mức GPU thì giảm còn 10 và ghi vào báo cáo.
NUM_WORKERS = min(4, os.cpu_count() or 2)
COMMON = dict(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR), out_dir=str(RUNS), pred_dir=str(PRED),
              curves_dir=str(CURVES), epochs=EPOCHS, num_workers=NUM_WORKERS)

def cfg(**kw):
    return Config(**{**COMMON, **kw})

def summary_of(exp_id, seed=0):
    p = RUNS / exp_id / f"seed{seed}" / "summary.json"
    return json.loads(p.read_text()) if p.exists() else None

def savefig(fig, name):
    fig.savefig(FIGS / name, dpi=120, bbox_inches="tight")
    plt.show()

## Bước 0 — EDA và kiểm tra pipeline

In [ ]:
train_df, val_df, test_df = D.load_split(LABELS_DIR, 0)
split_info = D.check_split(train_df, val_df, test_df, IMAGES_DIR)

pc = split_info["per_class"].copy()
pc["paper_table1"] = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
pc["khớp bài báo"] = pc["total"] == pc["paper_table1"]
pc.loc["TỔNG"] = pc.sum(numeric_only=True)
pc.to_csv(TABLES / "eda_split.csv")
display(pc)
print("Tỉ lệ lớp lớn nhất / nhỏ nhất (toàn bộ):", round(pc["total"][:-1].max() / pc["total"][:-1].min(), 2))

fig, ax = plt.subplots(figsize=(11, 4))
pc.iloc[:-1][["train", "val", "test"]].plot.bar(ax=ax)
ax.set(title="DeepWeeds fold 0 — số ảnh mỗi lớp theo tập", ylabel="số ảnh", xlabel="")
ax.grid(axis="y", alpha=.3); plt.xticks(rotation=30, ha="right")
savefig(fig, "eda_class_distribution.png")

In [ ]:
# >= 3 ảnh mỗi lớp + thống kê kích thước/kênh
from PIL import Image
fig, axes = plt.subplots(9, 4, figsize=(10, 22))
for c in range(9):
    files = train_df[train_df.Label == c].sample(4, random_state=0).Filename
    for j, f in enumerate(files):
        axes[c, j].imshow(Image.open(IMAGES_DIR / f)); axes[c, j].axis("off")
    axes[c, 0].set_title(D.CLASS_NAMES[c], loc="left", fontsize=10)
fig.tight_layout(); savefig(fig, "eda_samples.png")

sizes = pd.Series([Image.open(IMAGES_DIR / f).size + (Image.open(IMAGES_DIR / f).mode,)
                   for f in train_df.Filename.sample(300, random_state=0)]).value_counts()
print("Kích thước/kênh (300 ảnh ngẫu nhiên):\n", sizes)

In [ ]:
# Kiểm tra pipeline (GUIDE 1.3): seed, loss ban đầu ≈ ln 9, overfit một batch nhỏ, ảnh sau augmentation
c0 = cfg(backbone="resnet50", exp_id="CHECK")
l0 = train.initial_loss(c0)
print(f"Loss ban đầu (head mới) = {l0:.4f}  | kỳ vọng -ln(1/9) = {np.log(9):.4f}")

hist = train.overfit_one_batch(c0, n=16, steps=60)
print("Overfit 16 ảnh: loss đầu", round(hist[0], 4), "-> cuối", round(hist[-1], 4))
fig, ax = plt.subplots(1, 1, figsize=(5, 3)); ax.plot(hist); ax.set(title="Overfit 16 ảnh (resnet50)", xlabel="bước", ylabel="CE"); ax.set_yscale("log")
savefig(fig, "check_overfit_batch.png")
json.dump({"initial_loss": l0, "ln9": float(np.log(9)), "overfit_first": hist[0], "overfit_last": hist[-1]},
          open(TABLES / "pipeline_checks.json", "w"), indent=2)

# Ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn; thêm một batch CutMix
torch.manual_seed(0)
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for r, aug in enumerate(["basic", "trivial"]):
    ds = D.DeepWeedsDataset(train_df.sample(6, random_state=r), IMAGES_DIR, D.build_transforms(True, 224, aug))
    for j in range(6):
        x, y, f = ds[j]
        axes[r, j].imshow(D.denormalize(x).permute(1, 2, 0)); axes[r, j].set_title(f"{aug}: {D.CLASS_NAMES[y]}", fontsize=8); axes[r, j].axis("off")
ds = D.DeepWeedsDataset(train_df.sample(6, random_state=9), IMAGES_DIR, D.build_transforms(True, 224, "basic"))
xb = torch.stack([ds[j][0] for j in range(6)]); yb = torch.tensor(ds.labels)
xm, (ya, yb2, lam) = L.mix_batch(xb, yb, 1.0, "cutmix")
for j in range(6):
    axes[2, j].imshow(D.denormalize(xm[j]).permute(1, 2, 0)); axes[2, j].axis("off")
    axes[2, j].set_title(f"CutMix λ={lam:.2f}\n{D.CLASS_NAMES[ya[j]]} + {D.CLASS_NAMES[yb2[j]]}", fontsize=8)
fig.tight_layout(); savefig(fig, "check_augmentations.png")

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE 1.4: ImageNet pretrained, tinh chỉnh toàn bộ, RandomResizedCrop 224 + lật ngang, AdamW, LR 1e-4 / head 1e-3, wd 0.05 không áp cho norm/bias, warmup 1 epoch + cosine, CE, batch 64, AMP, chọn checkpoint theo macro-F1 val), **seed 0**, cùng split.

In [ ]:
BACKBONES = [
    ("B01", "resnet50"),                 # ResNet (mốc)
    ("B02", "resnext50_32x4d"),          # ResNeXt
    ("B03", "convnext_tiny"),            # ConvNeXt
    ("B04", "deit_small_patch16_224"),   # transformer
    ("B05", "efficientnet_b0"),          # nhẹ
    ("B06", "mobilenetv3_large_100"),    # nhẹ
]
bb_rows = []
for exp, name in BACKBONES:
    s = run(cfg(exp_id=exp, backbone=name, seed=0, desc=name))
    bb_rows.append(s)
    print(exp, name, "val F1", round(s["val_macro_f1"], 4))

df_bb = pd.DataFrame(bb_rows)
df_bb.to_csv(TABLES / "backbones.csv", index=False)
display(df_bb[["exp_id", "backbone", "weight_tag", "params_M", "gmacs", "val_macro_f1", "val_top1",
               "val_f1_chinee", "val_f1_snake", "best_epoch", "train_time_per_epoch_s", "latency_b1_p50_ms"]])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for _, r in df_bb.iterrows():
    ax[0].scatter(r.latency_b1_p50_ms, r.val_macro_f1, s=40 + 6 * r.params_M); ax[0].annotate(r.backbone, (r.latency_b1_p50_ms, r.val_macro_f1), fontsize=8)
    ax[1].scatter(r.gmacs, r.train_time_per_epoch_s); ax[1].annotate(r.backbone, (r.gmacs, r.train_time_per_epoch_s), fontsize=8)
ax[0].set(xlabel="độ trễ batch 1, p50 (ms, FP32, T4)", ylabel="macro-F1 val", title="Chất lượng vs độ trễ (kích thước điểm ~ #params)")
ax[1].set(xlabel="GMAC", ylabel="giây / epoch train", title="FLOPs có dự đoán thời gian train?")
for a in ax: a.grid(alpha=.3)
savefig(fig, "backbones_tradeoff.png")

# Chọn backbone đi tiếp: macro-F1 val cao nhất; nếu một backbone nhanh hơn kém < 0.005 (cỡ nhiễu 1 seed) thì ưu tiên backbone nhanh hơn.
ranked = df_bb.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
best = ranked.iloc[0]
cands = ranked[(best.val_macro_f1 - ranked.val_macro_f1 < 0.005) & (ranked.latency_b1_p50_ms < best.latency_b1_p50_ms)]
BB_ROW = cands.sort_values("latency_b1_p50_ms").iloc[0] if len(cands) else best
BB, BB_EXP = BB_ROW.backbone, BB_ROW.exp_id
print(f"Backbone đi tiếp: {BB} ({BB_EXP}) | val F1 {BB_ROW.val_macro_f1:.4f} | p50 {BB_ROW.latency_b1_p50_ms:.2f} ms")
print("Xếp hạng:\n", ranked[["backbone", "val_macro_f1", "latency_b1_p50_ms", "params_M"]])

## Bước 2 — Công thức huấn luyện (≥ 3 trục, mỗi lần khác `T00` đúng một yếu tố)
`T00` = công thức nền trên backbone đã chọn (seed 0 trùng đúng lần chạy ở Bước 1 nên dùng lại, ghi `alias_of`).
Chạy thêm `T00` seed 1, 2 ngay từ đây để **ước lượng nhiễu seed trên val** (std) dùng so sánh Δ (không đụng test).

In [ ]:
T00 = lambda seed: cfg(exp_id="T00", backbone=BB, seed=seed, desc="baseline")
t00 = [alias_run(cfg(exp_id=BB_EXP, backbone=BB, seed=0, desc=BB), T00(0))]
for s in (1, 2):
    t00.append(run(T00(s)))
f1_t00 = np.array([r["val_macro_f1"] for r in t00])
NOISE = float(f1_t00.std(ddof=1))
print("T00 val macro-F1 theo seed:", f1_t00.round(4), f"| mean {f1_t00.mean():.4f} ± {NOISE:.4f} (std, 3 seed)")

ABLATIONS = [  # (exp_id, trục, khác T00 ở điểm nào, thay đổi Config, desc)
    ("T01", "A. Khởi tạo",     "từ đầu (không pretrained)",           dict(init="scratch"),                 "scratch"),
    ("T02", "A. Khởi tạo",     "đóng băng backbone, chỉ train head",  dict(init="frozen"),                  "frozen"),
    ("T03", "B. Augmentation", "+ TrivialAugmentWide",                dict(aug="trivial"),                  "trivialaug"),
    ("T04", "B. Augmentation", "+ lật dọc",                           dict(aug="flipv"),                    "vflip"),
    ("T05", "B. Augmentation", "+ CutMix (α=1)",                      dict(mix="cutmix", mix_alpha=1.0),    "cutmix"),
    ("T06", "C. Loss",         "label smoothing ε=0.1",               dict(loss="ls", label_smoothing=0.1), "labelsmooth"),
    ("T07", "C. Loss",         "focal loss γ=2",                      dict(loss="focal", focal_gamma=2.0),  "focal"),
    ("T08", "C. Loss",         "CE có trọng số lớp 1/n_c (train)",    dict(loss="ce_weighted"),             "ce_weighted"),
    ("T09", "D. Cân bằng mẫu", "WeightedRandomSampler cân bằng lớp",  dict(sampler="balanced"),             "balanced_sampler"),
    ("T10", "F. Chính quy hoá", "EMA trọng số decay 0.998 (đánh giá bằng EMA)", dict(ema_decay=0.998),     "ema"),
]
abl_rows = []
for exp, axis, diff, change, desc in ABLATIONS:
    s = run(cfg(exp_id=exp, backbone=BB, seed=0, desc=desc, **change))
    abl_rows.append({**s, "axis": axis, "diff_vs_T00": diff, "change": json.dumps(change)})

In [ ]:
base0 = t00[0]["val_macro_f1"]
rows = [{**t00[0], "axis": "—", "diff_vs_T00": "công thức nền", "change": "{}"}] + abl_rows
df_tr = pd.DataFrame(rows)
df_tr["delta_vs_T00"] = df_tr["val_macro_f1"] - base0
df_tr["vs_noise"] = np.where(df_tr["delta_vs_T00"].abs() > NOISE, "vượt nhiễu", "≤ std: không phân biệt được")
df_tr.loc[df_tr.exp_id == "T00", "vs_noise"] = "—"
df_tr.to_csv(TABLES / "training.csv", index=False)
display(df_tr[["exp_id", "axis", "diff_vs_T00", "val_macro_f1", "val_top1", "delta_vs_T00", "vs_noise",
               "val_f1_chinee", "val_f1_snake", "best_epoch"]])
print(f"Nhiễu seed (std T00, 3 seed) = {NOISE:.4f}")

# Kết hợp tham lam: lấy ở mỗi trục (B, C, D, F) biến thể tốt nhất có Δ > std, rồi gộp lại (T11).
winners = []
for axis, g in df_tr[df_tr.exp_id != "T00"].groupby("axis"):
    if axis.startswith("A."):   # khởi tạo: tinh chỉnh toàn bộ là nền
        continue
    top = g.sort_values("val_macro_f1", ascending=False).iloc[0]
    if top.delta_vs_T00 > NOISE:
        winners.append(top)
if len(winners) < 2:  # không đủ yếu tố vượt nhiễu: thử gộp 2 yếu tố có Δ dương lớn nhất (ghi rõ trong báo cáo)
    pos = df_tr[(df_tr.exp_id != "T00") & (~df_tr.axis.str.startswith("A.")) & (df_tr.delta_vs_T00 > 0)]
    pos = pos.sort_values("delta_vs_T00", ascending=False).drop_duplicates("axis")
    winners = [r for _, r in pos.head(2).iterrows()]
COMBO = {}
for w in winners:
    COMBO.update(json.loads(w.change))
print("Kết hợp T11 từ:", [w.exp_id for w in winners], "->", COMBO)

In [ ]:
if len(winners) >= 2:
    s = run(cfg(exp_id="T11", backbone=BB, seed=0, desc="combo", **COMBO))
    df_tr = pd.concat([df_tr, pd.DataFrame([{**s, "axis": "Kết hợp", "diff_vs_T00": " + ".join(w.diff_vs_T00 for w in winners),
                                             "change": json.dumps(COMBO), "delta_vs_T00": s["val_macro_f1"] - base0}])], ignore_index=True)
    df_tr.loc[df_tr.exp_id == "T11", "vs_noise"] = np.where(abs(s["val_macro_f1"] - base0) > NOISE, "vượt nhiễu", "≤ std: không phân biệt được")
    df_tr.to_csv(TABLES / "training.csv", index=False)
    print("T11 val F1:", round(s["val_macro_f1"], 4), "| tổng các Δ đơn lẻ:", round(sum(w.delta_vs_T00 for w in winners), 4),
          "| Δ kết hợp:", round(s["val_macro_f1"] - base0, 4))

# Chốt công thức huấn luyện cho chung kết: cấu hình (không phải 'khởi tạo') có val macro-F1 seed 0 cao nhất
cand = df_tr[~df_tr.axis.str.startswith("A.")].sort_values("val_macro_f1", ascending=False).iloc[0]
FINAL_SRC = cand.exp_id
FINAL_CHANGE = json.loads(cand.change)
print(f"Công thức chung kết: {FINAL_SRC} {FINAL_CHANGE} | val F1 seed0 {cand.val_macro_f1:.4f} (T00 {base0:.4f})")

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ — chỉ trên **val**, không huấn luyện lại
Model: checkpoint seed 0 của công thức đã chốt (`FINAL_SRC`). Độ trễ: GPU T4, warmup 10, `torch.cuda.synchronize()` trước/sau, 100 lần đo, p50/p95/p99, **không tính tiền xử lý** (decode/resize JPEG), tạo view TTA tính trong thời gian.

In [ ]:
rd_inf = RUNS / FINAL_SRC / "seed0"
model_inf, cfg_inf = train.load_checkpoint(rd_inf, DEVICE)
cache_val = X.LoaderCache(val_df, IMAGES_DIR, num_workers=NUM_WORKERS)
METHODS = X.methods_for(cfg_inf.img_size)
TRANSFORMER = X.is_transformer(cfg_inf.backbone)

inf_rows, lat_rows, view_logits = [], [], {}
def add_latency(name, mdl, views_fn=None, img=224, dtype="fp32", fused=False):
    out = []
    for b in (1, 32):
        r = BM.latency_report(mdl, b, img, dtype, "cuda", warmup=10, iters=100 if b == 1 else 50,
                              fused_bn=fused, views_fn=views_fn, name=name)
        lat_rows.append(r); out.append(r)
    return out

for mid, m in METHODS.items():
    if m.cnn_only and TRANSFORMER:
        print("Bỏ qua", mid, "(ViT/DeiT/Swin không nhận độ phân giải khác 224)"); continue
    names_v, yv, lv = X.run_method(model_inf, m, cache_val, DEVICE, cfg_inf.img_size)
    view_logits[mid] = lv
    met = X.metrics(yv, X.probs_of(lv, m.space))
    img = int(m.input[3:]) if m.input.startswith("res") else (256 if m.input == "full" else 224)
    b1, b32 = add_latency(mid, model_inf, m.views, img)
    inf_rows.append({"exp_id": mid, "method": m.name, "model": f"{FINAL_SRC}/seed0 ({cfg_inf.backbone})", "K": m.k,
                     "val_macro_f1": met["macro_f1"], "val_top1": met["top1"], "val_ece": met["ece"],
                     "val_f1_chinee": met["f1"][0], "val_f1_snake": met["f1"][7],
                     "p50_ms": b1["p50"], "p95_ms": b1["p95"], "p99_ms": b1["p99"], "throughput_img_s_b32": b32["images_per_s"]})
    print(f"{mid:7s} F1 {met['macro_f1']:.4f} top1 {met['top1']:.4f} ECE {met['ece']:.4f} p50 {b1['p50']:.2f} ms")
Y_VAL = yv

In [ ]:
# I05 — Ensemble 3 backbone tốt nhất ở Bước 1 (trung bình xác suất; logit val đã lưu, cùng thứ tự file)
top3 = df_bb.sort_values("val_macro_f1", ascending=False).head(3)
probs = [X.probs_of([np.load(RUNS / e / "seed0" / "val_logits.npy")], "prob") for e in top3.exp_id]
assert all((np.load(RUNS / e / "seed0" / "val_labels.npy") == Y_VAL).all() for e in top3.exp_id)
met = X.metrics(Y_VAL, I.ensemble_probs(probs))
models = [train.load_checkpoint(RUNS / e / "seed0", DEVICE)[0] for e in top3.exp_id]
class Ens(torch.nn.Module):
    def __init__(s, ms): super().__init__(); s.ms = torch.nn.ModuleList(ms)
    def forward(s, x): return torch.stack([m(x).softmax(1) for m in s.ms]).mean(0)
b1, b32 = add_latency("I05", Ens(models))
inf_rows.append({"exp_id": "I05", "method": "Ensemble 3 backbone: " + ", ".join(top3.backbone), "model": ", ".join(top3.exp_id) + " (seed0)",
                 "K": 3, "val_macro_f1": met["macro_f1"], "val_top1": met["top1"], "val_ece": met["ece"],
                 "val_f1_chinee": met["f1"][0], "val_f1_snake": met["f1"][7],
                 "p50_ms": b1["p50"], "p95_ms": b1["p95"], "p99_ms": b1["p99"], "throughput_img_s_b32": b32["images_per_s"]})
del models

# I06 — Trọng số EMA (T10) so với T00, cùng seed 0, không tốn thêm khi suy luận
s10 = summary_of("T10")
inf_rows.append({"exp_id": "I06", "method": "Trọng số EMA (từ T10), 1 view", "model": "T10/seed0", "K": 1,
                 "val_macro_f1": s10["val_macro_f1"], "val_top1": s10["val_top1"], "val_ece": s10["val_ece"],
                 "val_f1_chinee": s10["val_f1_chinee"], "val_f1_snake": s10["val_f1_snake"],
                 "p50_ms": s10["latency_b1_p50_ms"], "p95_ms": s10["latency_b1_p95_ms"], "p99_ms": np.nan, "throughput_img_s_b32": np.nan})

# I07 — Temperature scaling trên I00: T khớp trên val; ECE sau đo (a) trên chính val (lạc quan) và (b) chéo 2 nửa val
z = view_logits["I00"][0]
T_val = I.fit_temperature(z, Y_VAL)
p_cal = I.apply_temperature(z, T_val)
met = X.metrics(Y_VAL, p_cal)
ece_before = X.metrics(Y_VAL, X.probs_of([z], "prob"))["ece"]
ece_cf = X.ece_crossfit(z, Y_VAL)
i00 = next(r for r in inf_rows if r["exp_id"] == "I00")
inf_rows.append({**i00, "exp_id": "I07", "method": f"I00 + temperature scaling (T={T_val:.3f}, khớp trên val)",
                 "val_ece": met["ece"], "val_ece_crossfit": ece_cf, "val_macro_f1": met["macro_f1"], "val_top1": met["top1"]})
print(f"Temperature T = {T_val:.3f} | ECE val trước {ece_before:.4f} -> sau {met['ece']:.4f} (cùng val) / {ece_cf:.4f} (chéo 2 nửa)")

In [ ]:
# I08 — Gộp BN vào conv (CNN có BatchNorm) và FP16 / AMP
fused = I.fuse_conv_bn(model_inf)
print("Số cặp conv+BN đã gộp:", fused.n_fused)
xchk = torch.randn(4, 3, 224, 224, device=DEVICE)
if fused.n_fused:
    print("Sai số lớn nhất trước/sau gộp:", I.max_abs_diff(model_inf, fused, xchk))
variants = [("I08a", "AMP (autocast FP16)", model_inf, "amp", False),
            ("I08b", "FP16 (model.half())", model_inf, "fp16", False)]
if fused.n_fused:
    variants += [("I08c", "Gộp BN, FP32", fused, "fp32", True), ("I08d", "Gộp BN + FP16", fused, "fp16", True)]
for mid, name, mdl, dtype, fz in variants:
    m_eval = mdl if dtype != "fp16" else __import__("copy").deepcopy(mdl).half()
    loader = cache_val.loader("crop")
    if dtype == "fp16":
        _, yv, lv = I.predict_views(m_eval, loader, DEVICE, lambda x: [x], amp=False, dtype=torch.float16)
    else:
        _, yv, lv = I.predict_views(m_eval, loader, DEVICE, lambda x: [x], amp=(dtype == "amp"))
    met = X.metrics(yv, X.probs_of(lv, "prob"))
    b1, b32 = add_latency(mid, mdl, None, 224, dtype, fz)
    inf_rows.append({"exp_id": mid, "method": name, "model": f"{FINAL_SRC}/seed0", "K": 1,
                     "val_macro_f1": met["macro_f1"], "val_top1": met["top1"], "val_ece": met["ece"],
                     "val_f1_chinee": met["f1"][0], "val_f1_snake": met["f1"][7],
                     "p50_ms": b1["p50"], "p95_ms": b1["p95"], "p99_ms": b1["p99"], "throughput_img_s_b32": b32["images_per_s"]})
if not fused.n_fused:
    print("Backbone không có BatchNorm (LayerNorm) -> gộp BN không áp dụng.")

df_inf = pd.DataFrame(inf_rows)
ref = df_inf.loc[df_inf.exp_id == "I00", "p50_ms"].item()
df_inf["cost_vs_I00"] = df_inf["p50_ms"] / ref
df_inf.to_csv(TABLES / "inference.csv", index=False)
df_lat = pd.DataFrame(lat_rows); df_lat.to_csv(TABLES / "latency.csv", index=False)
display(df_inf)

fig, ax = plt.subplots(figsize=(8, 5))
for _, r in df_inf.dropna(subset=["p50_ms"]).iterrows():
    ax.scatter(r.p50_ms, r.val_macro_f1); ax.annotate(r.exp_id, (r.p50_ms, r.val_macro_f1), fontsize=8)
ax.axvline(100, ls=":", c="r", label="ngân sách 100 ms"); ax.set_xscale("log")
ax.set(xlabel="độ trễ batch 1, p50 (ms, T4) — log", ylabel="macro-F1 val", title="Đánh đổi độ chính xác – độ trễ (val)")
ax.grid(alpha=.3); ax.legend(); savefig(fig, "inference_tradeoff.png")

In [ ]:
# Chốt phương pháp suy luận cho chung kết (một model, chỉ dựa trên val): F1 val cao nhất trong các phương pháp
# áp dụng được cho từng checkpoint (I00, I01*, I02*, I04_*). Kèm temperature scaling (khớp trên val).
single = df_inf[df_inf.exp_id.isin(list(METHODS))]
FINAL_METHOD = single.sort_values("val_macro_f1", ascending=False).iloc[0].exp_id
fm = single.set_index("exp_id").loc[FINAL_METHOD]
print(f"Suy luận chung kết: {FINAL_METHOD} — {fm.method} | val F1 {fm.val_macro_f1:.4f} | p95 {fm.p95_ms:.2f} ms")
rt = df_inf[df_inf.p95_ms <= 100].sort_values("val_macro_f1", ascending=False).iloc[0]
print(f"Tốt nhất khi p95 ≤ 100 ms: {rt.exp_id} — {rt.method} | val F1 {rt.val_macro_f1:.4f} | p95 {rt.p95_ms:.2f} ms")
json.dump({"FINAL_SRC": FINAL_SRC, "FINAL_CHANGE": FINAL_CHANGE, "FINAL_METHOD": FINAL_METHOD, "BB": BB,
           "realtime": rt.exp_id}, open(TABLES / "final_choice.json", "w"), indent=2)
del model_inf, fused; torch.cuda.empty_cache()

## Bước 4 — Chung kết: 3 seed, test **một lần mỗi seed**
- `F01` = backbone `BB` + công thức `FINAL_SRC` + suy luận `FINAL_METHOD` + temperature scaling (T khớp trên val). Seed 0 dùng lại lần chạy của `FINAL_SRC` (cùng cấu hình), seed 1, 2 huấn luyện mới.
- Mốc `T00` + `I00` với cùng 3 seed.
- Dự đoán: `predictions/F01_seed<k>_test.csv`, `F01_uncal_seed<k>_test.csv`, `F01_seed<k>_val.csv`, `T00_seed<k>_test.csv`.

In [ ]:
SEEDS = [0, 1, 2]
choice = json.load(open(TABLES / "final_choice.json"))
FINAL_SRC, FINAL_CHANGE, FINAL_METHOD = choice["FINAL_SRC"], choice["FINAL_CHANGE"], choice["FINAL_METHOD"]
F01 = lambda seed: cfg(exp_id="F01", backbone=BB, seed=seed, desc="final", **FINAL_CHANGE)
src0 = cfg(exp_id=FINAL_SRC, backbone=BB, seed=0, **FINAL_CHANGE)
f01 = [alias_run(src0, F01(0))] + [run(F01(s)) for s in SEEDS[1:]]

METHODS = X.methods_for(224)
final_info = []
for s in SEEDS:   # mỗi seed: test chạy đúng MỘT lần (nếu file đã có thì không chạy lại)
    if not (PRED / f"F01_seed{s}_test.csv").exists():
        final_info.append(X.final_predict(RUNS / "F01" / f"seed{s}", "F01", s, METHODS[FINAL_METHOD], PRED,
                                          IMAGES_DIR, LABELS_DIR, calibrate=True, device=DEVICE, num_workers=NUM_WORKERS))
    if not (PRED / f"T00_seed{s}_test.csv").exists():
        final_info.append(X.final_predict(RUNS / "T00" / f"seed{s}", "T00", s, METHODS["I00"], PRED,
                                          IMAGES_DIR, LABELS_DIR, calibrate=False, device=DEVICE, num_workers=NUM_WORKERS))
print(final_info)
!ls {PRED}

In [ ]:
# Chỉ số theo đúng định nghĩa của lớp (không sửa eval.py)
EV = REPO_DIR / "eval.py"; EVOUT = OUT / "eval_out"
for tag in ["F01", "F01_uncal", "T00"]:
    !python {EV} score --pred "{PRED}/{tag}_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {tag} --out {EVOUT}

In [ ]:
# Độ trễ p95 batch 1 của cấu hình chung kết (đo đúng cách, Bước 3) -> I5
P95_FINAL = float(df_inf.set_index("exp_id").loc[FINAL_METHOD, "p95_ms"]) if "df_inf" in globals() else \
            float(pd.read_csv(TABLES / "inference.csv").set_index("exp_id").loc[FINAL_METHOD, "p95_ms"])
P95_STR = f"{P95_FINAL:.2f}"
!python {EV} grade --final "{PRED}/F01_seed*_test.csv" --baseline "{PRED}/T00_seed*_test.csv" \
    --uncal "{PRED}/F01_uncal_seed*_test.csv" --final-val "{PRED}/F01_seed*_val.csv" \
    --latency-p95-ms {P95_STR} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {EVOUT} | tee {TABLES}/grade.txt

In [ ]:
# Bảng Final + PerClass từ chính file dự đoán (compute_metrics của eval.py)
def group_metrics(tag, split):
    out = []
    for s in SEEDS:
        p = read_pred(str(PRED / f"{tag}_seed{s}_{split}.csv"))
        out.append((s, compute_metrics(p.y_true, p.y_pred, p.probs)))
    return out

fin_rows = []
for tag, desc in [("F01", f"{BB} + {FINAL_SRC} {FINAL_CHANGE} + {FINAL_METHOD} + temperature scaling"),
                  ("T00", f"{BB} + T00 (công thức nền) + I00 (1 view)")]:
    te = dict(group_metrics(tag, "test")); va = dict(group_metrics(tag, "val"))
    unc = dict(group_metrics("F01_uncal", "test")) if tag == "F01" else {}
    for s in SEEDS:
        fin_rows.append({"exp_id": tag, "config": desc, "seed": s, "val_macro_f1": va[s]["macro_f1"],
                         "test_macro_f1": te[s]["macro_f1"], "test_top1": te[s]["top1"], "test_bal_acc": te[s]["balanced_acc"],
                         "test_ece": te[s]["ece"], "test_ece_uncal": unc[s]["ece"] if unc else np.nan,
                         "test_recall_chinee": te[s]["recall"][0], "test_recall_snake": te[s]["recall"][7],
                         "test_f1_chinee": te[s]["f1"][0], "test_f1_snake": te[s]["f1"][7]})
df_fin = pd.DataFrame(fin_rows)
agg_rows = []
for tag, g in df_fin.groupby("exp_id", sort=False):
    row = {"exp_id": tag, "config": g.config.iloc[0], "seed": f"mean ± std ({len(g)} seed, ddof=1)"}
    for c in g.columns[3:]:
        row[c] = f"{g[c].mean():.4f} ± {g[c].std(ddof=1):.4f}"
    agg_rows.append(row)
df_fin_all = pd.concat([df_fin.astype(object), pd.DataFrame(agg_rows)], ignore_index=True)
df_fin_all.to_csv(TABLES / "final.csv", index=False)
display(df_fin_all)

pc_rows = []
for tag in ["F01", "T00"]:
    ms = [m for _, m in group_metrics(tag, "test")]
    for c in range(9):
        pc_rows.append({"config": tag, "class": D.CLASS_NAMES[c], "n_test": int(ms[0]["support"][c]),
                        **{f"{k}_mean": float(np.mean([m[k][c] for m in ms])) for k in ("precision", "recall", "f1")},
                        **{f"{k}_std": float(np.std([m[k][c] for m in ms], ddof=1)) for k in ("precision", "recall", "f1")}})
df_pc = pd.DataFrame(pc_rows); df_pc.to_csv(TABLES / "perclass.csv", index=False)
display(df_pc)

In [ ]:
# Ma trận nhầm lẫn trên test (cộng 3 seed và chuẩn hoá theo hàng) + ảnh bị đoán sai Chinee Apple <-> Snake Weed (seed 0)
cm = sum(m["confusion"] for _, m in group_metrics("F01", "test"))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.1%}" if cm[i, j] else "", ha="center", va="center", fontsize=7,
                color="white" if cmn[i, j] > .5 else "black")
ax.set_xticks(range(9), D.CLASS_NAMES, rotation=40, ha="right"); ax.set_yticks(range(9), D.CLASS_NAMES)
ax.set(xlabel="dự đoán", ylabel="nhãn thật", title="F01 — ma trận nhầm lẫn test (tổng 3 seed; số ảnh và % theo hàng)")
fig.colorbar(im); savefig(fig, "final_confusion_matrix.png")

p0 = read_pred(str(PRED / "F01_seed0_test.csv"))
pairs = [(0, 7), (7, 0)]
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for r, (t, pdx) in enumerate(pairs):
    idx = np.where((p0.y_true == t) & (p0.y_pred == pdx))[0][:6]
    for j in range(6):
        axes[r, j].axis("off")
        if j < len(idx):
            k = idx[j]
            axes[r, j].imshow(Image.open(IMAGES_DIR / p0.filenames[k]))
            axes[r, j].set_title(f"thật {D.CLASS_NAMES[t]}\nđoán {D.CLASS_NAMES[pdx]} ({p0.probs[k, pdx]:.2f})", fontsize=8)
fig.suptitle("Ảnh test bị nhầm giữa Chinee Apple và Snake Weed (F01 seed 0)"); fig.tight_layout()
savefig(fig, "final_errors_chinee_snake.png")
print("Nhầm lớn nhất ngoài đường chéo:")
off = [(cm[i, j], D.CLASS_NAMES[i], D.CLASS_NAMES[j]) for i in range(9) for j in range(9) if i != j]
for n, a, b in sorted(off, reverse=True)[:6]:
    print(f"  {a} -> {b}: {n}")

## Bước 5 — Sản phẩm: `results.xlsx`, `curves/`, `predictions/`, bảng cho báo cáo

In [ ]:
df_bb = pd.read_csv(TABLES / "backbones.csv"); df_tr = pd.read_csv(TABLES / "training.csv")
df_inf = pd.read_csv(TABLES / "inference.csv"); df_lat = pd.read_csv(TABLES / "latency.csv")
df_fin_all = pd.read_csv(TABLES / "final.csv"); df_pc = pd.read_csv(TABLES / "perclass.csv")

sh_bb = df_bb.assign(resolution=224, epochs=EPOCHS, note=np.where(df_bb.exp_id == BB_EXP, "đi tiếp sang Bước 2-3", ""))[
    ["exp_id", "backbone", "weight_tag", "params_M", "gmacs", "resolution", "epochs", "seed", "val_macro_f1", "val_top1",
     "val_f1_chinee", "val_f1_snake", "train_time_per_epoch_s", "latency_b1_p50_ms", "latency_b1_p95_ms", "note"]].rename(
    columns={"params_M": "params (M)", "gmacs": "GMAC", "train_time_per_epoch_s": "train time/epoch (s)",
             "latency_b1_p50_ms": "latency b1 p50 (ms)", "latency_b1_p95_ms": "latency b1 p95 (ms)"})
sh_tr = df_tr.assign(note=df_tr.get("alias_of", pd.Series([""] * len(df_tr))).fillna("").map(lambda a: f"cùng lần chạy {a}" if a else ""))[
    ["exp_id", "backbone", "axis", "diff_vs_T00", "seed", "val_macro_f1", "val_top1", "delta_vs_T00", "vs_noise",
     "val_f1_chinee", "val_f1_snake", "best_epoch", "note"]].rename(columns={"delta_vs_T00": "Δ macro-F1 vs T00"})
noise_row = {"exp_id": "T00 (3 seed)", "backbone": BB, "axis": "nhiễu", "diff_vs_T00": "mean val macro-F1 / std (ddof=1) của T00 qua seed 0,1,2",
             "seed": "0,1,2", "val_macro_f1": f1_t00.mean() if "f1_t00" in globals() else np.nan,
             "Δ macro-F1 vs T00": NOISE if "NOISE" in globals() else np.nan, "vs_noise": "std"}
sh_tr = pd.concat([sh_tr, pd.DataFrame([noise_row])], ignore_index=True)
sh_inf = df_inf.rename(columns={"p50_ms": "p50 b1 (ms)", "p95_ms": "p95 b1 (ms)", "p99_ms": "p99 b1 (ms)",
                                "throughput_img_s_b32": "throughput b32 (img/s)", "cost_vs_I00": "chi phí tương đối vs I00"})
sh_lat = df_lat[["config", "gpu", "dtype", "batch", "img_size", "k_views", "fused_bn", "p50", "p95", "p99", "images_per_s", "torch", "n_iters", "warmup"]].rename(
    columns={"p50": "p50 (ms)", "p95": "p95 (ms)", "p99": "p99 (ms)", "images_per_s": "img/s"})
summ = pd.concat([
    df_bb.assign(kind="backbone", cost=df_bb.latency_b1_p50_ms)[["exp_id", "kind", "backbone", "val_macro_f1", "val_top1", "cost"]],
    df_tr.assign(kind="training")[["exp_id", "kind", "backbone", "val_macro_f1", "val_top1"]].assign(cost=np.nan),
    df_inf.assign(kind="inference", backbone=BB, cost=df_inf.p50_ms)[["exp_id", "kind", "backbone", "val_macro_f1", "val_top1", "cost"]],
]).rename(columns={"cost": "latency b1 p50 (ms)"}).drop_duplicates("exp_id").sort_values("val_macro_f1", ascending=False).head(10)
fin_agg = df_fin_all[df_fin_all.seed.astype(str).str.startswith("mean")][["exp_id", "config", "test_macro_f1", "test_top1", "test_ece", "test_recall_chinee", "test_recall_snake"]]
summary_sheet = pd.concat([summ.assign(section="Top 10 theo macro-F1 val"),
                           fin_agg.rename(columns={"config": "backbone"}).assign(section="Chung kết vs mốc (TEST, mean ± std 3 seed)")], ignore_index=True)

X.write_xlsx({"Summary": summary_sheet, "Backbones": sh_bb, "Training": sh_tr, "Inference": sh_inf, "Final": df_fin_all,
              "PerClass": df_pc, "Latency": sh_lat},
             OUT / "results.xlsx",
             highlight={"Backbones": "val_macro_f1", "Training": "val_macro_f1", "Inference": "val_macro_f1"})
print("Đã ghi", OUT / "results.xlsx")

# Bảng markdown để dán vào report.md
md_parts = ["## Backbones\n", X.df_to_md(sh_bb), "\n\n## Training\n", X.df_to_md(sh_tr), "\n\n## Inference\n", X.df_to_md(sh_inf),
            "\n\n## Final\n", X.df_to_md(df_fin_all), "\n\n## PerClass\n", X.df_to_md(df_pc), "\n\n## eval.py grade\n```\n",
            (TABLES / "grade.txt").read_text() if (TABLES / "grade.txt").exists() else "", "```\n"]
(OUT / "report_tables.md").write_text("".join(md_parts), encoding="utf-8")

# Kiểm tra mỗi thí nghiệm huấn luyện có ảnh curves/
exp_ids = set(df_bb.exp_id) | set(df_tr.exp_id) | {"F01"}
have = {p.name.split("_")[0] for p in CURVES.glob("*.png")}
print("Thiếu ảnh đường cong:", sorted(exp_ids - have) or "không")

In [ ]:
# Chép sản phẩm vào thư mục bài nộp trong repo + nén để tải về
for sub in ["curves", "predictions", "figures"]:
    shutil.copytree(OUT / sub, SUB_DIR / sub, dirs_exist_ok=True)
for f in ["results.xlsx", "report_tables.md"]:
    shutil.copy(OUT / f, SUB_DIR / f)
shutil.copytree(TABLES, SUB_DIR / "tables", dirs_exist_ok=True)
shutil.make_archive(str(DRIVE_ROOT / f"{MSSV}_{HO_TEN}_outputs"), "zip", SUB_DIR)
print("Đã chép vào", SUB_DIR, "và nén ->", DRIVE_ROOT / f"{MSSV}_{HO_TEN}_outputs.zip")
if IN_COLAB:
    from google.colab import files
    files.download(str(DRIVE_ROOT / f"{MSSV}_{HO_TEN}_outputs.zip"))
# Push từ Colab (tuỳ chọn): cần GitHub token có quyền repo
# !cd {REPO_DIR} && git add submissions/{MSSV}_{HO_TEN} && git commit -m "Add results" && git push https://<TOKEN>@github.com/{GITHUB_USER}/{REPO_NAME}.git